# Ultimate Pandas Guide: Housing Data Analysis

This notebook is a comprehensive reference for **Pandas** methods, applied to the King County Housing dataset. 
It covers:
1. **Input/Output**: Reading and writing data.
2. **Inspection**: Viewing data structure and types.
3. **Selection & Indexing**: Grabbing specific rows/cols (`loc`, `iloc`).
4. **Data Cleaning**: Handling missing values, duplicates, and types.
5. **Math & Statistics**: Aggregations, correlations, describing data.
6. **Grouping & Sorting**: `groupby`, `pivot_table`, `sort_values`.
7. **Advanced Operations**: `apply`, `map`, string methods, and merging.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization setup
sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Input / Output (I/O)
Methods for loading and saving data.

In [ ]:
# read_csv(): Load data from a CSV file
df = pd.read_csv('kc_house_data.csv')

# Other common loaders (examples):
# df_excel = pd.read_excel('file.xlsx')
# df_json = pd.read_json('file.json')

print("Data Loaded Successfully!")

In [ ]:
# to_csv(): Write data to a CSV file
# index=False prevents writing the row numbers as a column
df.head(10).to_csv('sample_output.csv', index=False)

## 2. DataFrame Inspection
Methods to understand the shape and structure of your data.

In [ ]:
# head(n): View first n rows (default 5)
df.head()

In [ ]:
# tail(n): View last n rows
df.tail()

In [ ]:
# info(): Concise summary (index dtype, columns, non-null values, memory usage)
df.info()

In [ ]:
# shape: Tuple containing dimensions (rows, columns)
print(f"Shape: {df.shape}")

# columns: Index of column names
print(f"Columns: {df.columns.tolist()}")

# index: Index of the rows
print(f"Index: {df.index}")

# dtypes: Data type of each column
print(f"\nData Types:\n{df.dtypes.head(3)}")

## 3. Selection & Indexing
Crucial for slicing data. 
- `loc`: Label-based selection.
- `iloc`: Integer-position based selection.

In [ ]:
# Selecting a specific column (Returns Series)
prices = df['price']

# Selecting multiple columns (Returns DataFrame)
subset = df[['id', 'price', 'bedrooms']]
subset.head(3)

In [ ]:
# iloc: Select by Position (Rows 0 to 4, Columns 0 to 2)
df.iloc[0:5, 0:3]

In [ ]:
# loc: Select by Label (All rows, specific named columns)
df.loc[0:5, ['price', 'sqft_living', 'grade']]

In [ ]:
# Conditional Selection (Filtering)
# Example: Houses with 5+ bedrooms AND price < 500k
big_cheap_houses = df[(df['bedrooms'] >= 5) & (df['price'] < 500000)]
print(f"Found {len(big_cheap_houses)} houses matching criteria.")
big_cheap_houses[['price', 'bedrooms', 'sqft_living']].head()

In [ ]:
# query(): A readable way to filter data string expressions
df.query("waterfront == 1 and grade > 10")

In [ ]:
# isin(): Filter by a list of values
# Example: Only look at houses with exactly 2 or 4 bedrooms
df[df['bedrooms'].isin([2, 4])].head()

## 4. Data Cleaning
Handling "dirty" data (nulls, duplicates, wrong types).

In [ ]:
# isnull() / isna(): Detect missing values
print(df.isnull().sum())

In [ ]:
# dropna(): Remove rows with missing values
df_clean = df.dropna()

# fillna(value): Fill missing values with a constant or calculated value (like mean)
# df['price'] = df['price'].fillna(df['price'].mean()) 

In [ ]:
# astype(): Convert data types
# Example: Convert 'floors' from float to int (truncating decimal)
df['floors_int'] = df['floors'].astype(int)
df[['floors', 'floors_int']].head()

In [ ]:
# rename(): Rename columns
df_renamed = df.rename(columns={'dates': 'date_sold', 'price': 'sale_price'})
df_renamed.columns[:5]

In [ ]:
# drop(): Remove columns or rows
# axis=1 means columns, axis=0 means rows
df_dropped = df.drop(columns=['id', 'lat', 'long'])
df_dropped.head(2)

In [ ]:
# duplicates(): Handle duplicates
print(f"Duplicates found: {df.duplicated().sum()}")
df = df.drop_duplicates()

## 5. Math & Statistics
Pandas wraps NumPy visualization and calculation methods.

In [ ]:
# describe(): Summary stats for numerical columns
df[['price', 'sqft_living']].describe()

In [ ]:
# Individual Aggregate Functions
print(f"Max Price: {df['price'].max()}")
print(f"Min Price: {df['price'].min()}")
print(f"Mean Price: {df['price'].mean()}")
print(f"Median Price: {df['price'].median()}")
print(f"Mode Bedrooms: {df['bedrooms'].mode()[0]}")

In [ ]:
# value_counts(): Count unique values (Histogram-like)
df['views_count'] = df['view'].value_counts()
print("Counts of houses per view rating:")
print(df['view'].value_counts())

In [ ]:
# corr(): Correlation matrix
df[['price', 'sqft_living', 'grade', 'yr_built']].corr()

## 6. Grouping and Sorting
Powerful analysis often requires splitting data into groups.

In [ ]:
# groupby(): Group by a categorical column and aggregate
# Example: Average price for each number of bedrooms
avg_price_by_bed = df.groupby('bedrooms')['price'].mean()
avg_price_by_bed.sort_values(ascending=False)

In [ ]:
# Multiple Aggregations
df.groupby('waterfront')['price'].agg(['mean', 'min', 'max', 'count'])

In [ ]:
# sort_values(): Sort dataframe
# Top 5 most expensive houses
df.sort_values(by='price', ascending=False).head(5)

In [ ]:
# pivot_table(): Excel-style pivot tables
# Avg price by view (index) and waterfront (columns)
df.pivot_table(values='price', index='view', columns='waterfront', aggfunc='median')

## 7. Advanced Manipulations
Apply, String Methods, Time Series, and Merging.

In [ ]:
# apply(): Apply a function to every row/column
# Example: Label houses as 'Expensive' or 'Affordable'
def label_price(price):
    if price > 1000000:
        return 'Expensive'
    else:
        return 'Affordable'

df['price_category'] = df['price'].apply(label_price)
df['price_category'].value_counts()

In [ ]:
# string methods (.str): Use string functions on columns
# Convert 'date' string column to proper datetime manually (if it wasn't already)
# note: our date col is already datetime, so converting back to str for demo
df['date_str'] = df['date'].astype(str)
df['year_extracted'] = df['date_str'].str[:4] # Extract first 4 chars
df[['date', 'year_extracted']].head()

In [ ]:
# Time Series Accessors (.dt)
# Note: df['date'] must be datetime objects first
df['month'] = df['date'].dt.month_name()
df['day_of_week'] = df['date'].dt.day_name()
df[['date', 'month', 'day_of_week']].head()

In [ ]:
# concat(): Combining DataFrames vertically
part1 = df.iloc[:5]
part2 = df.iloc[5:10]
combined = pd.concat([part1, part2])
combined

In [ ]:
# merge(): SQL-style joins
# Creating dummy data for demonstration
locations = pd.DataFrame({'zipcode': [98178, 98125], 'city': ['Seattle', 'Seattle']})
merged_df = pd.merge(df, locations, on='zipcode', how='left')
merged_df[['zipcode', 'city']].head()

## 8. Basic Visualization (Pandas & Seaborn)

In [ ]:
# plotting directly from pandas
df['price'].plot(kind='box', figsize=(8, 6), title='Price Boxplot')
plt.show()

In [ ]:
# Scatter plot with Seaborn
plt.figure(figsize=(10, 6))
sns.regplot(x='sqft_living', y='price', data=df.sample(2000), line_kws={'color':'red'})
plt.title('Sqft vs Price (Line of Best Fit)')
plt.show()